# Framework 3/4 — SCM Fitting, Root Cause, Interventions & Counterfactuals

> **Thesis:** Toward Actionable Root Cause Analysis: Integrating Causal Discovery and
> Counterfactual XAI for Predictive Maintenance
>
> **Prototype dataset:** NASA C-MAPSS FD001 · **Notebook 3 of 4** · 2026-07-30

The framework has nine modules. This notebook covers **Module 5 (SCM fitting), Module 6 (root cause analysis), Module 7 (interventions) and Module 8 (counterfactual XAI)**.

All settings come from `pipeline_config.py` in the repo root. All logic lives in
`src/causalpdm/framework/`.

## Setup

In [1]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT))

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)

import pipeline_config as cfg
from causalpdm.framework import artifacts

from causalpdm.framework import counterfactual, intervention, rca, scm

prep = artifacts.load("prep")
feat = artifacts.load("feat")
edges = artifacts.load("edges_final")

variables = ([v for v in cfg.CAUSAL_VARIABLES if v in prep["feature_cols"]]
             if cfg.CAUSAL_VARIABLES else prep["feature_cols"])
print(f"Loaded {len(edges)} causal edges over {len(variables)} variables")

Loaded 8 causal edges over 11 variables


## Module 5 — SCM fitting (mechanism & noise estimation)

### Why this step must exist

The causal graph says *which* sensors influence which. It says nothing about *how much*, or in
what form. A graph alone cannot answer "what if we changed this?" — for that we need a
**structural causal model**: one equation per variable, plus the noise that the equations do not
explain.

For each variable, we fit:

$$X_i(t) = f_i\big(\text{parents of } X_i\big) + e_i(t)$$

The noise term $e_i(t)$ is not a nuisance — it is essential. Pearl's counterfactual procedure
needs it, because it is what makes an explanation specific to *one engine* rather than to the
fleet average. Without this module, Modules 7 and 8 are impossible.

### Two deliberate choices

- **Linear mechanisms.** For a prototype this is the right call: identifiable given the graph,
  fast, and auditable. The code accepts any scikit-learn regressor, so a nonlinear mechanism can
  be swapped in later without touching Modules 6–8.
- **Fitted on the trended data.** The mechanisms must live in the same space as the health
  indicator and all simulations. This is where the two-version design from notebook 1 pays off.
- **An autoregressive term is added** to every equation. Sensors are strongly autocorrelated; if
  we omitted each sensor's own history, that variance would be wrongly attributed to its parents
  and inflate the apparent causal effects.

In [2]:
scm_result = artifacts.cached("scm", lambda: scm.run(
    prep["normalised"], variables, edges, tau_max=cfg.MAX_LAG
))
model = scm_result["scm"]

print(f"Equations fitted   : {scm_result['n_mechanisms']}")
print(f"Exogenous variables: {scm_result['n_exogenous']} (no parents, so no equation)")
print(f"Average fit (R2)   : {scm_result['mean_r2']:.3f}")
print(f"\nEvaluation order (contemporaneous links must be acyclic to simulate):")
print(" -> ".join(model.topological_order))

Equations fitted   : 11
Exogenous variables: 0 (no parents, so no equation)
Average fit (R2)   : 0.487

Evaluation order (contemporaneous links must be acyclic to simulate):
s_2 -> s_3 -> s_4 -> s_7 -> s_8 -> s_11 -> s_12 -> s_15 -> s_17 -> s_20 -> s_21


In [3]:
summary = scm_result["summary"]
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.barh(summary["variable"], summary["r2"], color="steelblue")
ax.set_xlabel("R² of the fitted structural equation")
ax.set_title("How well each sensor is explained by its causal parents")
ax.set_xlim(0, 1)
plt.tight_layout(); plt.show()
summary

,variable,n_parents,r2,noise_std,parents
0,s_11,4,0.737068,0.511747,"s_7(t-2), s_15(t-1), s_7(t-1), s_11(t-1)"
1,s_4,2,0.652262,0.588414,"s_11(t-2), s_4(t-1)"
2,s_12,1,0.584503,0.643537,s_12(t-1)
3,s_21,2,0.536092,0.679497,"s_11(t-1), s_21(t-1)"
4,s_7,1,0.502234,0.703879,s_7(t-1)
5,s_20,2,0.498544,0.706559,"s_7(t-2), s_20(t-1)"
6,s_2,2,0.461582,0.732675,"s_12(t-2), s_2(t-1)"
7,s_15,1,0.413081,0.764651,s_15(t-1)
8,s_8,1,0.397453,0.775032,s_8(t-1)
9,s_17,1,0.316270,0.824913,s_17(t-1)


A high R² means that sensor is well explained by its parents; a low one means most of its
behaviour is in its own noise term. Both are useful — a low-R² variable is a poor *target* for
prediction but can still be an excellent *root cause*, because it acts on the system from outside.

## Module 6 — Root cause analysis

### The problem this solves

Near failure, almost every sensor looks abnormal at once. If you rank by "how abnormal", you get
the loudest **symptom**, not the **cause** — and a maintenance engineer sent to fix a symptom
fixes nothing. Correlation genuinely cannot solve this. Direction is what separates them, and
direction is exactly what the causal graph provides.

### How the ranking works

1. **Measure deviation** for each sensor: pre-failure window against *this engine's own* healthy
   baseline. A within-engine comparison, so fleet variation cannot masquerade as a cause.
2. **Trace causal paths**: for each deviating sensor, which *other* deviating sensors can it reach
   in the graph?
3. **Separate causes from symptoms**: if a deviating sensor has another deviating sensor as an
   ancestor, it is downstream — a symptom. If nothing deviating explains it, it is a root cause
   candidate.
4. **Rank** by deviation magnitude × causal reach, with symptoms demoted.

In [4]:
rca_result = rca.run(feat["featured"], variables, edges, unit=cfg.ENGINE_TO_ANALYSE)

print(f"Engine analysed   : {rca_result['unit']}")
print(f"Degraded at cycle : {rca_result['onset_cycle']} of {rca_result['total_cycles']}")
print(f"Root causes found : {len(rca_result['root_causes'])}")
print(f"Symptoms found    : {len(rca_result['symptoms'])}")

Engine analysed   : 39
Degraded at cycle : 119 of 128
Root causes found : 5
Symptoms found    : 5


In [5]:
print("Full ranking — note how the verdict column separates causes from symptoms:")
rca_result["ranking"][["variable", "abs_deviation", "causal_reach",
                       "n_symptoms_explained", "is_root_cause", "root_cause_score"]]

Full ranking — note how the verdict column separates causes from symptoms:


,variable,abs_deviation,causal_reach,n_symptoms_explained,is_root_cause,root_cause_score
0,s_7,3.403881,0.531941,4,True,5.214544
1,s_8,4.055292,0.000000,0,True,4.055292
2,s_12,3.516997,0.124880,1,True,3.956200
3,s_15,2.294885,0.345798,3,True,3.088452
4,s_3,2.092395,0.000000,0,True,2.092395
5,s_11,4.620490,0.376001,2,False,1.589450
6,s_4,3.563288,0.000000,0,False,0.890822
7,s_20,2.506962,0.000000,0,False,0.626740
8,s_21,2.200776,0.000000,0,False,0.550194
9,s_2,1.948494,0.000000,0,False,0.487123


In [6]:
dev = rca_result["deviations"].head(10).iloc[::-1]
roots = set(rca_result["root_causes"]["variable"])
colours = ["#dc2626" if v in roots else "#94a3b8" for v in dev["variable"]]

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.barh(dev["variable"], dev["deviation_z"], color=colours)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("deviation from this engine's healthy baseline (standard deviations)")
ax.set_title("Red = identified root cause · grey = symptom or unremarkable")
plt.tight_layout(); plt.show()

### The causal chains behind the verdict

This is what makes the output *explainable* rather than a black-box ranking: each root cause comes
with the actual paths through which it produces the observed symptoms.

In [7]:
for _, row in rca_result["root_causes"].iterrows():
    print(f"\n{row['variable']}  (deviation {row['abs_deviation']:.2f} sd, "
          f"explains {int(row['n_symptoms_explained'])} symptoms)")
    for path in row["explained_symptoms"]:
        print(f"    {path}")


s_7  (deviation 3.40 sd, explains 4 symptoms)
    s_7 -> s_11 -> s_4
    s_7 -> s_20
    s_7 -> s_11 -> s_21
    s_7 -> s_11

s_8  (deviation 4.06 sd, explains 0 symptoms)

s_12  (deviation 3.52 sd, explains 1 symptoms)
    s_12 -> s_2

s_15  (deviation 2.29 sd, explains 3 symptoms)
    s_15 -> s_11 -> s_4
    s_15 -> s_11 -> s_21
    s_15 -> s_11

s_3  (deviation 2.09 sd, explains 0 symptoms)


## Module 7 — Intervention analysis (do-operations)

### The question

*If we had held this sensor in its healthy range, what would have happened to engine health?*

Formally, for a candidate cause $V$ and health readout $H$:

$$\text{effect}(V) = E[H \mid do(V = v_{\text{safe}})] - E[H]$$

We clamp $V$ in the fitted SCM and re-simulate forward through the mechanisms. The health score is
then **recomputed from the simulated sensors** — so the effect on health is derived, never assumed.
This is precisely why the health indicator was built as a readout function instead of a graph node.

### Scope — important for the thesis

On C-MAPSS these are **model-based estimates that cannot be validated.** The dataset contains no
interventional data, so there is no ground truth to check them against. The same code run on
causRCA *can* be validated, against its 100 labelled do-interventions. That contrast is the
argument for causRCA being the thesis centrepiece.

In [8]:
candidates = rca_result["root_causes"]["variable"].tolist()
if not candidates:
    candidates = rca_result["ranking"]["variable"].tolist()
candidates = candidates[: cfg.INTERVENTION_CANDIDATES]
print(f"Testing interventions on: {candidates}")

interv = intervention.run(
    model, rca_result["engine"], feat["health_readout"],
    candidates, rca_result["onset_idx"], lead_time=cfg.MAINTENANCE_LEAD_TIME,
)
print(f"\nMost effective lever: {interv['best_variable']}")
print(f"Maintenance assumed {cfg.MAINTENANCE_LEAD_TIME} cycles before onset "
      f"(from cycle index {interv['start_idx']})")

Testing interventions on: ['s_7', 's_8', 's_12', 's_15']

Most effective lever: s_7
Maintenance assumed 20 cycles before onset (from cycle index 98)


In [ ]:
interv["ranking"]

In [9]:
if not interv["dose_response"].empty:
    curve = interv["dose_response"]
    fig, ax = plt.subplots(figsize=(8, 3.2))
    ax.plot(curve["do_value"], curve["health_intervened"], "o-", color="#1f6feb", lw=2)
    ax.set_xlabel(f"value we force {interv['best_variable']} to hold")
    ax.set_ylabel("resulting health loss")
    ax.set_title(f"Dose-response curve for do({interv['best_variable']})")
    plt.tight_layout(); plt.show()
    print("A sloped curve supports a 'keep this sensor below X' recommendation.")
    print("A flat curve would mean this sensor is not a useful lever at any value —")
    print("which is itself an actionable finding: replace the component instead.")

A sloped curve supports a 'keep this sensor below X' recommendation.
A flat curve would mean this sensor is not a useful lever at any value —
which is itself an actionable finding: replace the component instead.


## Module 8 — Counterfactual XAI

### Interventions vs counterfactuals

Module 7 asked a *general* question: what happens if we clamp this variable. Module 8 asks the
retrospective, engine-specific question a maintenance engineer actually asks:

> *"This engine failed. Would it have failed if sensor X had stayed in range?"*

Answering that requires Pearl's three-step procedure, implemented literally:

| Step | What happens | Why it matters |
|---|---|---|
| **Abduction** | Recover this engine's own noise $e(t)$ from its observed trajectory | Makes the answer about *this* engine, not the fleet |
| **Action** | Modify the model with $do(X = x')$ | The hypothetical change |
| **Prediction** | Re-simulate with that noise **held fixed** | The twin world differs *only* through the intervention |

Holding the noise fixed is the whole point. Without it you would be comparing this engine to an
average engine, not to itself.

In [10]:
cf = counterfactual.run(
    model, rca_result["engine"], feat["health_readout"],
    interv["best_variable"], interv["start_idx"],
    feat["failure_threshold"], unit=rca_result["unit"],
)

print("COUNTERFACTUAL EXPLANATION")
print("=" * 78)
print(cf["explanation"])

COUNTERFACTUAL EXPLANATION
Engine 39 reached a degraded state. Had s_7 been held above 0.60 from cycle 99 onward (a shift of 1.8 standard deviations from its actual level of -1.21), the health indicator would have peaked at 0.62 instead of 0.83, staying below the 0.65 failure threshold.


### Searching for the *minimal* change

An explanation demanding a physically impossible value is useless as a maintenance instruction. So
the search sweeps the engine's own observed range, nearest value first, and stops at the smallest
change that avoids failure. Anything it finds is achievable by construction.

In [11]:
print("Values tried, nearest to the engine's actual behaviour first:")
display(cf["attempts"])
print(f"\nActual level was {cf['factual_level']:.2f}; "
      f"failure threshold is {cf['threshold']:.2f}")

Values tried, nearest to the engine's actual behaviour first:


,do_value,valid,proximity_abs,proximity_sd,sparsity,plausible,health_reduction
0,-1.293441,False,0.085553,0.085889,1,True,0.026217
1,-0.820408,False,0.387480,0.389003,1,True,0.071140
2,-1.766473,False,0.558585,0.560780,1,True,-0.018706
3,-0.347375,False,0.860513,0.863894,1,True,0.116062
4,-2.239506,False,1.031618,1.035671,1,True,-0.063629
5,0.125657,False,1.333546,1.338785,1,True,0.160985
6,0.598690,True,1.806578,1.813677,1,True,0.205908



Actual level was -1.21; failure threshold is 0.65


### Scoring the explanation

Counterfactual XAI has standard quality criteria, and a thesis should report all of them rather
than only showing a nice-looking result:

- **Validity** — does the change actually avoid the failure?
- **Proximity** — how big a change was needed? (smaller is better)
- **Sparsity** — how many variables had to move? (fewer is better)
- **Plausibility** — is the required value inside the engine's real operating range?

In [12]:
if cf["found"]:
    m = cf["best"]["metrics"]
    r = cf["best"]["result"]
    print(f"Valid        : {m['valid']}")
    print(f"Proximity    : {m['proximity_sd']:.2f} standard deviations")
    print(f"Sparsity     : {m['sparsity']} variable changed")
    print(f"Plausible    : {m['plausible']} (value lies within observed range)")
    print(f"Health cut   : {m['health_reduction']:.3f}")

    fig, ax = plt.subplots(figsize=(10, 3.4))
    ax.plot(r["health_factual"].to_numpy(), color="#dc2626", lw=2,
            label="factual — engine degraded")
    ax.plot(r["health_counterfactual"].to_numpy(), color="#059669", lw=2,
            label=f"counterfactual — do({r['variable']} = {r['cf_value']:.2f})")
    ax.axhline(feat["failure_threshold"], color="black", ls="--", lw=1, label="failure threshold")
    ax.axvline(r["start_idx"], color="#d97706", ls=":", lw=1.5, label="intervention point")
    ax.set_xlabel("cycle index"); ax.set_ylabel("health loss")
    ax.set_title("The twin world: identical engine, identical noise, one variable changed")
    ax.legend(fontsize=8, frameon=False)
    plt.tight_layout(); plt.show()
else:
    print("No achievable single-variable change avoided failure for this engine.")
    print("That is a legitimate result: it points to component replacement rather than")
    print("an operating-point adjustment.")

Valid        : True
Proximity    : 1.81 standard deviations
Sparsity     : 1 variable changed
Plausible    : True (value lies within observed range)
Health cut   : 0.206


## Save for the dashboard notebook

In [13]:
artifacts.save("rca", rca_result)
artifacts.save("interv", interv)
artifacts.save("cf", cf)
print("Saved: rca, interv, cf")
print("\nNext: 04_actionable_dashboard — assemble everything into the decision-support view.")

Saved: rca, interv, cf

Next: 04_actionable_dashboard — assemble everything into the decision-support view.


## What this notebook established

| Module | Result |
|---|---|
| 5 — SCM fitting | Mechanisms and noise estimated; interventions and counterfactuals now possible |
| 6 — Root cause | Causes separated from symptoms using graph direction, with explicit causal paths |
| 7 — Intervention | Candidate levers ranked by estimated effect on health; dose-response curve produced |
| 8 — Counterfactual | Engine-specific explanation via abduction/action/prediction, scored on the standard criteria |

**The honest limitation:** Modules 7 and 8 produce estimates that C-MAPSS cannot validate. This is
not a coding gap — it is a property of the dataset, and it is the reason the thesis moves to
causRCA for its validated results.